In [1]:
import torch
import torch as nn
from torch.utils.data import Dataset,DataLoader


In [2]:
# Dataset class adapted from https://github.com/isaaccorley/torchrs 
import os
import json
from typing import List, Dict

import torch
import torchvision.transforms as T
from PIL import Image


class RSICD(torch.utils.data.Dataset):
    """ Image Captioning Dataset from 'Exploring Models and Data for
    Remote Sensing Image Caption Generation', Lu et al. (2017)
    https://arxiv.org/abs/1712.07835

    'RSICD is used for remote sensing image captioning task. more than ten thousands
    remote sensing images are collected from Google Earth, Baidu Map, MapABC, Tianditu.
    The images are fixed to 224X224 pixels with various resolutions. The total number of
    remote sensing images are 10921, with five sentences descriptions per image.'
    """
    splits = ["train", "val", "test"]

    def __init__(
        self,
        root: str = "data",
        split: str = "train",
        transform: T.Compose = T.Compose([T.ToTensor()])
    ):
        assert split in self.splits
        self.root = root
        self.transform = transform
        self.captions = self.load_captions(os.path.join(root, "dataset_rsicd.json"), split)
        self.image_root = "RSICD_images"

    @staticmethod
    def load_captions(path: str, split: str) -> List[Dict]:
        with open(path) as f:
            captions = json.load(f)["images"]
        return [c for c in captions if c["split"] == split]

    def __len__(self) -> int:
        return len(self.captions)

    def __getitem__(self, idx: int) -> Dict:
        captions = self.captions[idx]
        path = os.path.join(self.root, self.image_root, captions["filename"])
        x = Image.open(path).convert("RGB")
        x = self.transform(x)
        sentences = [sentence["raw"] for sentence in captions["sentences"]]
        return dict(x=x, captions=sentences)






In [6]:
import torch
from torch import nn
from torchvision import models

efficient_netb0 = models.efficientnet_b0(pretrained=True)


class Encoder(nn.Module):
    def __init__(self,encoded_dim,projection_dim):
        super().__init__()
        self.encoder = nn.Sequential(*list(efficient_netb0.children())[:-1])
        self.fc = nn.Linear(encoded_dim,projection_dim) # project to same dim as decoder embedding


    def forward(self,images):
        features = self.encoder(images) #(B,1280,1,1)
        features = features.flatten(start_dim =1)
        #features = features.view(features.size(0),-1) # (B,1280)
        features = self.fc(features)

        return features


c:\Users\Acer nitro\anaconda3\envs\rsicd\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and will be removed in 0.15, please use 'weights' instead.
  warnings.warn(
c:\Users\Acer nitro\anaconda3\envs\rsicd\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and will be removed in 0.15. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [7]:
import torch
from torch import nn

class Decoder(nn.Module):
    def __init__(self,embed_dim,hidden_dim,vocab_size,num_layers = 1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size,embed_dim)
        self.lstm = nn.LSTM(embed_dim,hidden_dim,num_layers,batch_first = True)
        self.fc = nn.Linear(hidden_dim,vocab_size)

    def forward(self, captions, features):
        # captions: (B, T)
        embeddings = self.embedding(captions)  # (B, T, embed_dim)
        embeddings = torch.cat((features.unsqueeze(1), embeddings), dim=1)  # prepend image feature
        outputs, _ = self.lstm(embeddings)  # (B, T+1, hidden_dim)
        outputs = self.fc(outputs)  # (B, T+1, vocab_size)
        return outputs

In [8]:
class ImageCaptioningModel(nn.Module):
    def __init__(self,Encoder,Decoder):
        super().__init__()
        self.encoder = Encoder
        self.decoder = Decoder
        
    def forward(self,image,captions):
        features = self.encoder(image)
        output = self.decoder(features,captions)

In [ ]:
Encoded_dim = 1280 # this becuase of efficientNet
Projected_dim = 512

In [ ]:
encoder = Encoder(encoded_dim=Encoded_dim,projection_dim=Projected_dim)
decoder = Decoder(embed_dim=Projected_dim,hidden_dim=512,vocab_size=)

In [ ]:
base_model = ImageCaptioningModel(Encoder,Decoder)
EPOCHS = 50

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(base_model.parameters())